In [1]:
# !pip install spacy_transformers
# !pip install -U spacy

In [1]:
import spacy
from spacy.tokens import DocBin
from tqdm import tqdm
import json
import transformers

In [3]:
spacy.__version__

'3.7.4'

In [4]:
# !nvidia-smi

In [5]:
# !git clone https://github.com/laxmimerit/CV-Parsing-using-Spacy-3

In [7]:
import json

# Open the JSON file and load its content
with open('D:\\Nirma University\\Trimesters\\Trimester 9\\Internship\\Internship Project\\CV-Parsing-using-Spacy-3-master\\data\\training\\train_data.json') as file:
    cv_data = json.load(file)


In [8]:
len(cv_data)

200

In [10]:
!python -m spacy init fill-config data/training/base_config.cfg data/training/config.cfg

✔ Auto-filled config with all values
✔ Saved config
data\training\config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [11]:
cv_data[0]

['Govardhana K Senior Software Engineer  Bengaluru, Karnataka, Karnataka - Email me on Indeed: indeed.com/r/Govardhana-K/ b2de315d95905b68  Total IT experience 5 Years 6 Months Cloud Lending Solutions INC 4 Month • Salesforce Developer Oracle 5 Years 2 Month • Core Java Developer Languages Core Java, Go Lang Oracle PL-SQL programming, Sales Force Developer with APEX.  Designations & Promotions  Willing to relocate: Anywhere  WORK EXPERIENCE  Senior Software Engineer  Cloud Lending Solutions -  Bangalore, Karnataka -  January 2018 to Present  Present  Senior Consultant  Oracle -  Bangalore, Karnataka -  November 2016 to December 2017  Staff Consultant  Oracle -  Bangalore, Karnataka -  January 2014 to October 2016  Associate Consultant  Oracle -  Bangalore, Karnataka -  November 2012 to December 2013  EDUCATION  B.E in Computer Science Engineering  Adithya Institute of Technology -  Tamil Nadu  September 2008 to June 2012  https://www.indeed.com/r/Govardhana-K/b2de315d95905b68?isid=rex-

In [12]:
def get_spacy_doc(file,data):
  nlp=spacy.blank('en')
  db = DocBin()

  for text,annot in tqdm(data):
    doc = nlp.make_doc(text)
    annot = annot['entities']

    ents =[]
    entity_indices = []

    for start, end, label in annot:
      skip_entity = False
      for idx in range(start, end):
        if idx in entity_indices:
          skip_entity = True
          break
      if skip_entity == True:
        continue

      entity_indices = entity_indices + list(range(start,end))

      try:
        span = doc.char_span(start , end , label = label, alignment_mode = 'contract')
      except:
        continue

      if span is None:
        err_data = str([start , end]) +"    " + str(text) + "\n"
        file.write(err_data)

      else:
        ents.append(span)

    try:
      doc.ents = ents
      db.add(doc)
    except:
      pass

  return db


In [13]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(cv_data, test_size=0.3)

In [14]:
len(train), len(test)

(140, 60)

In [23]:
file = open('error.txt', 'w',encoding='utf-8')

db = get_spacy_doc(file, train)
db.to_disk('train_data.spacy')

db = get_spacy_doc(file, test)
db.to_disk('test_data.spacy')

file.close()

100%|██████████| 60/60 [00:01<00:00, 50.81it/s]


In [24]:
len(db.tokens)

60

In [25]:
!python -m spacy train data/training/config.cfg --output ./output --paths.train ./train_data.spacy --paths.dev ./test_data.spacy

ℹ Saving to output directory: output
ℹ Using CPU

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  -------------  --------  ------  ------  ------  ------
  0       0         855.75   1509.24    0.22    0.12    4.53    0.00
⚠ Aborting and saving the final best model. Encountered exception:
ValueError("[E024] Could not find an optimal move to supervise the parser.
Usually, this means that the model can't be updated in a way that's valid and
satisfies the correct annotations specified in the GoldParse. For example, are
all labels added to the model? If you're training a named entity recognizer,
also make sure that none of your annotated entity spans have leading or trailing
whitespace or punctuation. You can also use t

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Traceback (most recent call last):
  File "c:\Program Files\Python39\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Program Files\Python39\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\user\AppData\Roaming\Python\Python39\site-packages\spacy\__main__.py", line 4, in <module>
    setup_cli()
  File "C:\Users\user\AppData\Roaming\Python\Python39\site-packages\spacy\cli\_util.py", line 87, in setup_cli
    command(prog_name=COMMAND)
  File "C:\Users\user\AppData\Roaming\Python\Python39\site-packages\click\core.py", line 1130, in __call__
    return self.main(*args, **kwargs)
  File "C:\Users\user\AppData\Roami

In [20]:
!python -m spacy debug data data/training/config.cfg


============================ Data file validation ============================
✔ Pipeline can be initialized with data
✔ Corpus is loadable

=============================== Training stats ===============================
Language: en
Training pipeline: transformer, ner
140 training docs
60 evaluation docs
⚠ 1 training examples also in evaluation data
⚠ Low number of examples to train a new pipeline (140)

============================== Vocab & Vectors ==============================
ℹ 87053 total word(s) in the data (10364 unique)
ℹ No word vectors present in the package

========================== Named Entity Recognition ==========================
ℹ 11 label(s)
0 missing value(s) (tokens with '-' label)
✘ 58 invalid whitespace entity spans
⚠ Low number of examples for label 'Years of Experience' (27)

⠙ Analyzing label distribution...
⠹ Analyzing label distribution...
⠸ Analyzing label distribution...
⠼ Analyzing label distribution...
⚠ Low number of examples for label 'UNKNOWN' (2)



Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
nlp = spacy.load('')

In [ ]:
doc = nlp('')
for ent in doc.ents:
  print(ent.text, "   ->>>>>",ent.label_)

In [ ]:
!pip install PyMuPDF

In [ ]:
import sys, fitz

In [ ]:
fname = ''
doc = fitz.open(fname)

In [ ]:
doc

In [ ]:
text = " "
for page in doc:
  text = text + str(page.get_text())

In [ ]:
text = text.strip()

In [ ]:
text = ' '.join(text.split())

In [ ]:
text

In [ ]:
doc = nlp(text)
for ent in doc.ents:
  print(ent.text, "   ->>>>> ",ent.label_)